In [1]:
import pandas as pd
import os
from tqdm.auto import tqdm
from llm_asr_clarification.constants import SAMPLE_MEETINGS
import json
import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer
from transformers import AutoTokenizer
from jiwer import wer
import jiwer
import re
import string
from collections import Counter

# # Define a robust transformation pipeline
# # This applies data cleaning steps in order, from top to bottom
# JIWER_TRANSFORM = jiwer.Compose([
#     jiwer.ToLowerCase(),                # Convert all text to lowercase
#     jiwer.RemovePunctuation(),          # Strip characters like commas, periods, question marks
#     jiwer.RemoveMultipleSpaces(),       # Turn multi-spaces into a single space
#     jiwer.Strip(),                      # Clean up leading/trailing whitespaces
#     jiwer.ReduceToListOfListOfWords()   # Format text tokens perfectly for jiwer's internal engine
# ])


pd.set_option('display.max_colwidth', None)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"   # or another causal LM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    # use_stemmer=True,
    use_stemmer=False
)

def rouge_l(pred, ref):
    return scorer.score(
        ref,   # reference first
        pred   # prediction second
    )["rougeL"]

def load_df_from_path(AMI_PATH):
    meeting_paths = [entry.path for entry in os.scandir(AMI_PATH)]
        
    data = []
    for meeting_path in tqdm(meeting_paths):
        beam_results = os.path.join(meeting_path, "artifacts", "beam_results.json")
        try:
            with open(beam_results, "r", encoding="utf-8") as f:
                lines = f.read()
            lines = json.loads(lines)
        except Exception as err:
            print(f"couldnt open file {beam_results}")
            continue
    
        # Process lines
        for line in lines:
            for i in range(1,6):
                beam_no = f'beam_{i}'
                beam = line.pop(beam_no)
    
                line[f"{beam_no}_text"] = beam["text"]
                line[f"{beam_no}_asrlogprob"] = beam["asr_avg_log_prob"]
                line[f"{beam_no}_llmlogprob"] = beam["llm_avg_log_prob"]
            line["meeting_name"] = meeting_path.split("/")[-1]

            data.append(line)
            
    # df = pd.DataFrame(lines)
    df = pd.DataFrame(data)
    return df
    
AMI_TRAIN_PATH = '/group/jrwhitehill/amicorpus/train'
AMI_VAL_PATH = '/group/jrwhitehill/amicorpus/validation'

df_train = load_df_from_path(AMI_TRAIN_PATH)
df_val = load_df_from_path(AMI_VAL_PATH)

print(df_train.shape)
print(df_val.shape)

  0%|          | 0/136 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

(88717, 17)
(10590, 17)


In [3]:
df_train.head(3)

,gt,beam_1_text,beam_1_asrlogprob,beam_1_llmlogprob,beam_2_text,beam_2_asrlogprob,beam_2_llmlogprob,beam_3_text,beam_3_asrlogprob,beam_3_llmlogprob,beam_4_text,beam_4_asrlogprob,beam_4_llmlogprob,beam_5_text,beam_5_asrlogprob,beam_5_llmlogprob,meeting_name
0,Okay.,Okay.,-0.165992,-10.1875,Okay.,-0.165992,-10.18750,Okay.,-0.165992,-10.1875,OK.,-0.808041,-11.0000,Okay.,-0.165992,-10.1875,ES2005d
1,"Okay, almost there.",Okay.,-0.202354,-10.1875,Okay. I want to say.,-0.761428,-5.21875,Okay.,-0.202354,-10.1875,Okay.,-0.202354,-10.1875,Okay. I was there.,-0.371616,-5.5000,ES2005d
2,Okay.,Okay,-0.634828,-17.0000,Ok,-2.403044,-19.62500,Okay.,-0.134573,-10.1875,OK.,-0.716196,-11.0000,Okay.,-0.134573,-10.1875,ES2005d


# Helper Methods

In [4]:
def find_common_words_from_ASR(df: pd.DataFrame, text_column: str, top_n: int = 100):
    """Finds the most common short utterances in your dataset."""
    short_utterances = []
    
    for text in df[text_column]:
        if not isinstance(text, str):
            continue
            
        # Clean the text by Removing punctuation.
        clean_text = re.sub(r'[^\w\s]', '', text.lower()).strip()
        word_count = len(clean_text.split())
        
        # Look for 1 - 5 word utterances
        if 1 <= word_count <= 5:
            short_utterances.append(clean_text)
            
    # Find the most common ones
    common_phrases = Counter(short_utterances).most_common(top_n)

    # Return the most common words
    common_words = set()
    for phrase, count in common_phrases:
        common_words |= set(phrase.split())

    return common_words

# Find most common words from training DF
common_words_from_ASR = find_common_words_from_ASR(df_train, 'beam_1_text')
print(common_words_from_ASR)
print(len(common_words_from_ASR))

{'maybe', 'think', 'know', 'on', 'the', 'of', 'can', 'um', 'dont', 'this', 'in', 'well', 'go', 'a', 'haha', 'we', 'is', 'yes', 'mmm', 'you', 'uh', 'very', 'screen', 'god', 'okay', 'wow', 'nice', 'yeah', 'cool', 'then', 'thats', 'next', 'what', 'ah', 'so', 'right', 'thanks', 'great', 'or', 'please', 'no', 'bye', 'alright', 'for', 'exactly', 'because', 'something', 'here', 'mean', 'and', 'one', 'im', 'now', 'luck', 'again', 'but', 'to', 'sure', 'thank', 'ok', 'time', 'much', 'really', 'good', 'yep', 'true', 'like', 'i', 'oh', 'my', 'mm', 'sorry', 'mmhmm', 'there', 'all', 'that', 'it', 'its', 'hmm', 'watching'}
80


In [5]:
import nltk
from nltk.corpus import stopwords

# Download standard NLTK stopwords if you haven't already
nltk.download('stopwords', quiet=True)

# Define complete set of stop words
standard_stops = set(stopwords.words('english'))
# all_stop_words = standard_stops
all_stop_words = standard_stops.union(common_words_from_ASR)

def normalize_and_remove_stopwords(text: str) -> str:
    """
    Cleans text, removes stop words, and returns the normalized string.
    Returns an empty string if nothing meaningful is left.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
        
    # Lowercase the text
    text = text.lower()
    
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    # Split into words and check against stop words
    words = text.split()
    meaningful_words = [w for w in words if w not in all_stop_words]
    
    # Reconstruct and return the normalized string
    return " ".join(meaningful_words)

# Process all columns of DF

In [6]:
df_train.shape

(88717, 17)

In [7]:
df_val.shape

(10590, 17)

In [13]:
def process_columns(df):
    llm_logprob_columns = [f'beam_{i}_llmlogprob' for i in range(1,6)]
    asr_logprob_columns = [f'beam_{i}_asrlogprob' for i in range(1,6)]
    
    llm_logprobs = torch.tensor(df[llm_logprob_columns].values)
    asr_logprobs = torch.tensor(df[asr_logprob_columns].values)
    
    # # ALL BEAMS
    # scores = F.softmax(ALPHA*llm_logprobs + 0.0asr_logprobs, dim=1)
    # highest_score_idxs = torch.argmax(scores, dim=1, keepdim=True)
    # highest_scores = torch.gather(scores, dim=1, index=highest_score_idxs)

    # JUST BEAM 1
    scores = llm_logprobs
    highest_scores = scores[torch.arange(scores.size(0)), torch.zeros(scores.size(0), dtype=torch.long)]


    new_df = df.copy()

    # AGGREGATED STATS
    new_df['max_llmlogprobs'] = torch.max(llm_logprobs, dim=1).values.numpy()
    new_df['min_llmlogprobs'] = torch.min(llm_logprobs, dim=1).values.numpy()
    new_df['spread_llmlogprobs'] = new_df['max_llmlogprobs'] - new_df['min_llmlogprobs']

    new_df['max_asrlogprobs'] = torch.max(asr_logprobs, dim=1).values.numpy()
    new_df['min_asrlogprobs'] = torch.min(asr_logprobs, dim=1).values.numpy()
    new_df['spread_asrlogprobs'] = new_df['max_asrlogprobs'] - new_df['min_asrlogprobs']

    new_df['highest_score'] = highest_scores.numpy()

    # Add a new column for generated text using beam_1_text
    new_df['text'] = new_df['beam_1_text']

    # Calculate number of tokens for beam_1_text and GT
    new_df['num_tokens_text'] = new_df['beam_1_text'].apply(
        lambda x: len(tokenizer.encode(x))
    )
    new_df['num_tokens_gt'] = new_df['gt'].apply(
        lambda x: len(tokenizer.encode(x))
    )

    # Normalize and Remove Stop words from text and gt columns
    new_df['text_norm'] = new_df['text'].apply(normalize_and_remove_stopwords)
    new_df['gt_norm'] = new_df['gt'].apply(normalize_and_remove_stopwords)

    # Keep only the rows where the ground truth is NOT an empty string.
    # The .copy() ensures we don't get SettingWithCopyWarnings later.
    new_df = new_df[new_df['gt_norm'] != ""].copy()
    
    
    new_df["rougeL"] = [
        rouge_l(pred, ref).fmeasure
        for pred, ref in zip(new_df["text_norm"], new_df["gt_norm"])
    ]
    new_df["wer"] = [
        min(1.0, wer(reference = ref, hypothesis = pred))
        for pred, ref in zip(new_df["text_norm"], new_df["gt_norm"])
    ]

    return new_df

In [14]:
df = process_columns(df_train)

In [15]:
df.shape

(56671, 31)

In [17]:
df.head(10)

,gt,beam_1_text,beam_1_asrlogprob,beam_1_llmlogprob,beam_2_text,beam_2_asrlogprob,beam_2_llmlogprob,beam_3_text,beam_3_asrlogprob,beam_3_llmlogprob,...,min_asrlogprobs,spread_asrlogprobs,highest_score,text,num_tokens_text,num_tokens_gt,text_norm,gt_norm,rougeL,wer
1,"Okay, almost there.",Okay.,-0.202354,-10.18750,Okay. I want to say.,-0.761428,-5.21875,Okay.,-0.202354,-10.18750,...,-0.761428,0.559074,-10.18750,Okay.,3,6,,almost,0.000000,1.00
3,"We'll sta I'll use the PowerPoint, I guess.","Oh well, I'll use the PowerPoint, I guess.",-0.256419,-4.90625,"well, I'll use the powerpoint I guess.",-0.460730,-6.03125,"Well, I'll use the PowerPoint, I guess.",-0.103038,-5.28125,...,-0.460730,0.357692,-4.90625,"Oh well, I'll use the PowerPoint, I guess.",13,13,ill use powerpoint guess,sta ill use powerpoint guess,0.888889,0.20
4,"How was that, was that fun?","How was that, was that?",-0.126335,-6.53125,"I was that, was that.",-0.454225,-8.18750,I was that was enough.,-0.371302,-9.18750,...,-0.454225,0.327890,-6.53125,"How was that, was that?",8,9,,fun,0.000000,1.00
7,Very fun.,you,-0.053533,-20.50000,you,-0.053533,-20.50000,you,-0.053533,-20.50000,...,-0.053533,0.000000,-20.50000,you,2,4,,fun,0.000000,1.00
9,"Uh oh I've forgotten to mail you the minutes, but I will do.",Oh forgot to mail you a minute but I will do.,-0.493088,-6.34375,"Oh, I forgot to mail you the minute, but I will do.",-0.201260,-4.81250,oh I forgot to mail you a minute but I will do.,-0.281761,-5.68750,...,-0.493088,0.291827,-6.34375,Oh forgot to mail you a minute but I will do.,13,17,forgot mail minute,ive forgotten mail minutes,0.285714,0.75
11,Upsidaisy.,"Oops, it's Daisy.",-0.422295,-6.34375,"Oops, Daisy.",-0.052387,-8.37500,Oops Daisy.,-0.358518,-11.06250,...,-0.422295,0.369908,-6.34375,"Oops, it's Daisy.",7,5,oops daisy,upsidaisy,0.000000,1.00
14,E excuse me I forgot my copy.,"Yeah, not right okay",-1.284902,-7.90625,and we're good,-1.709243,-11.00000,"we, yeah, good way. Alright, okay.",-0.659660,-6.46875,...,-1.709243,1.440533,-7.90625,"Yeah, not right okay",6,9,,e excuse forgot copy,0.000000,1.00
16,He's gonna get his pen.,you,-0.053533,-20.50000,you,-0.053533,-20.50000,you,-0.053533,-20.50000,...,-1.225393,1.171860,-20.50000,you,2,8,,hes gonna get pen,0.000000,1.00
19,Um Will you guys first with your prototype um before we get to the good news?,"Okay. Well, you guys, first of all, you put the type before we get to the good news.",-0.262269,-4.21875,"Okay. Well you guys, first of all you prototype before we get to the good news.",-0.248296,-4.78125,"Okay. Well, you guys, first of all, you prototype. Before we get to the good news...",-0.390336,-4.28125,...,-80.333649,80.156204,-4.21875,"Okay. Well, you guys, first of all, you put the type before we get to the good news.",24,18,guys first put type get news,guys first prototype get news,0.727273,0.40
20,"Yeah, there's good news?","Yeah, that's good news",-0.296412,-4.90625,"Yeah, that's good news.",-0.108904,-4.34375,"Yeah, that's good news.",-0.108904,-4.34375,...,-0.296412,0.187508,-4.90625,"Yeah, that's good news",7,8,news,theres news,0.666667,0.50
